# 121 -- Structured Path C: Network-Substructure (Layer 1 analog)

**Blueprint:** `outputs/reports/structured/structured_path_c_blueprint.md` -- read this first.
**Mirrors:** DS5's `notebooks/101_ds5_network_subphenotype.ipynb` (Path C), reframed: DS5 asked "is there
substructure within the NeT-dominated hub-locked profile?"; Structured's Phase 1 is NOT NeT-dominated
(confirmed by Path A: ARI vs Phase1 = 1.000 using behavioral features alone), so this notebook instead
tests the complementary question -- does NeT alone explain anything, and is there NeT substructure the
dominant behavioral split doesn't already capture?

**Output root:** `outputs/model_outputs/structured/phenotyping/path_c/`


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings("ignore")

MO = PROJECT_ROOT / "outputs" / "model_outputs" / "structured"
OUT = MO / "phenotyping" / "path_c"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
print("Output folder:", OUT)

def bootstrap_jaccard(X_z, labels_full, k, n_boot=100, frac=0.8, seed=RANDOM_STATE):
    n = len(X_z)
    rng = np.random.RandomState(seed)
    jaccards = []
    for b in range(n_boot):
        idx = rng.choice(n, size=max(int(n * frac), k * 2), replace=False)
        km_sub = KMeans(n_clusters=k, n_init=10, random_state=seed + b).fit(X_z[idx])
        labels_sub = km_sub.labels_
        labels_full_sub = labels_full[idx]
        cont = np.zeros((k, k))
        for i in range(k):
            for j in range(k):
                cont[i, j] = np.sum((labels_full_sub == i) & (labels_sub == j))
        row_ind, col_ind = linear_sum_assignment(-cont)
        pair_jaccards = []
        for i, j in zip(row_ind, col_ind):
            set_full = set(np.where(labels_full_sub == i)[0])
            set_sub = set(np.where(labels_sub == j)[0])
            union = len(set_full | set_sub)
            pair_jaccards.append(len(set_full & set_sub) / union if union > 0 else np.nan)
        jaccards.append(np.nanmean(pair_jaccards))
    return float(np.mean(jaccards)), float(np.std(jaccards))

## 1. Population and features (Part 1 of blueprint)

In [ ]:
net = pd.read_csv(MO / "network_theory" / "participant_network_metrics.csv")
intt = pd.read_csv(MO / "information_theory" / "participant_entropy_metrics.csv")
phase1 = pd.read_csv(MO / "phenotyping" / "cluster_assignments.csv")
patha = pd.read_csv(MO / "phenotyping" / "path_a" / "behavior_only_cluster_assignments.csv")

ne = net[net["is_net_eligible"]].copy()
hsm = intt.set_index("participant_id")["H_state_marginal"]
ne["H_state_marginal"] = ne["participant_id"].map(hsm)

feature_cols = ["HHI_net", "rho_self", "H_trans", "H_state_marginal"]
print(f"N (NeT-eligible): {len(ne)}")
print(f"H_state_marginal non-null: {ne['H_state_marginal'].notna().sum()}")
assert len(ne) == 45 and ne["H_state_marginal"].notna().sum() == 45

corr = ne[feature_cols + ["top1_share_net"]].corr()
print()
print(corr.round(3))
print()
print(f"top1_share_net vs HHI_net: r={corr.loc['top1_share_net','HHI_net']:.3f} -- excluded, near-collinear")
print(f"H_trans vs H_state_marginal: r={corr.loc['H_trans','H_state_marginal']:.3f} -- flagged, retained (conceptually distinct)")

ne[["participant_id"] + feature_cols].to_csv(OUT / "net_only_feature_matrix.csv", index=False)

## 2. Full-sample NeT-only k-selection (N=45)

In [ ]:
X = ne[feature_cols].to_numpy()
X_z = (X - X.mean(axis=0)) / X.std(axis=0)

MIN_CELL = 10
k_results = {}
for k in range(2, 5):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X_z)
    labels = km.labels_
    sizes = pd.Series(labels).value_counts()
    sil = silhouette_score(X_z, labels)
    ch = calinski_harabasz_score(X_z, labels)
    db = davies_bouldin_score(X_z, labels)
    k_results[k] = {"labels": labels, "silhouette": sil, "CH": ch, "DB": db,
                     "min_size": int(sizes.min()), "sizes": sizes.to_dict()}
    print(f"k={k}: silhouette={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, min_cell={sizes.min()}, sizes={dict(sizes)}")

valid_ks = [k for k, r in k_results.items() if r["min_size"] >= MIN_CELL]
print(f"\nValid k (min_cell >= {MIN_CELL}): {valid_ks}")
SELECTED_K = max(valid_ks, key=lambda k: k_results[k]["silhouette"]) if valid_ks else 2
print(f"Selected k: {SELECTED_K}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
ks = list(k_results.keys())
axes[0].plot(ks, [k_results[k]["silhouette"] for k in ks], marker="o")
axes[0].set_title("Silhouette"); axes[0].set_xlabel("k")
axes[1].plot(ks, [k_results[k]["CH"] for k in ks], marker="o", color="#55A868")
axes[1].set_title("Calinski-Harabasz"); axes[1].set_xlabel("k")
axes[2].plot(ks, [k_results[k]["DB"] for k in ks], marker="o", color="#C44E52")
axes[2].set_title("Davies-Bouldin"); axes[2].set_xlabel("k")
for ax in axes:
    ax.axvline(SELECTED_K, color="gray", ls="--", lw=1)
plt.tight_layout()
plt.savefig(FIG / "fig1_pathc_k_selection.png", dpi=150)
plt.show()
print("Saved fig1_pathc_k_selection.png")

In [ ]:
jac_mean, jac_std = bootstrap_jaccard(X_z, k_results[SELECTED_K]["labels"], SELECTED_K)
stability_verdict = "STABLE" if jac_mean >= 0.75 else "MARGINAL" if jac_mean >= 0.60 else "UNSTABLE"
print(f"Bootstrap Jaccard (k={SELECTED_K}): mean={jac_mean:.3f}, std={jac_std:.3f} -> {stability_verdict}")

pca = PCA(n_components=len(feature_cols))
pca.fit(X_z)
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f"  PC{i+1}: {v*100:.1f}%")
print(f"Cumulative PC1+PC2: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 6))
coords = pca.transform(X_z)[:, :2]
labels_sel = k_results[SELECTED_K]["labels"]
for lab in np.unique(labels_sel):
    mask = labels_sel == lab
    ax.scatter(coords[mask, 0], coords[mask, 1], label=f"Cluster {lab} (n={mask.sum()})", alpha=0.7)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title(f"Path C (NeT-only) PCA biplot, k={SELECTED_K}")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "fig2_pathc_pca_biplot.png", dpi=150)
plt.show()
print("Saved fig2_pathc_pca_biplot.png")

In [ ]:
PROFILE_NAMES = [f"NetProfile {chr(65+i)}" for i in range(SELECTED_K)]
label_to_name = {lab: PROFILE_NAMES[i] for i, lab in enumerate(sorted(np.unique(labels_sel)))}
ne["cluster_label"] = [label_to_name[l] for l in labels_sel]

ne[["participant_id", "cluster_label"] + feature_cols].to_csv(OUT / "net_only_cluster_assignments.csv", index=False)
cluster_profiles = ne.groupby("cluster_label")[feature_cols].agg(["mean", "median", "std"])
cluster_profiles.to_csv(OUT / "net_only_cluster_profiles.csv")

print("Cluster sizes:"); print(ne["cluster_label"].value_counts())
print(); print("Cluster means (raw):"); print(ne.groupby("cluster_label")[feature_cols].mean().round(4))

fig, ax = plt.subplots(figsize=(8, 5))
means = ne.groupby("cluster_label")[feature_cols].mean()
means_z = (means - ne[feature_cols].mean()) / ne[feature_cols].std()
means_z.T.plot(kind="bar", ax=ax)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("z-score relative to NeT-eligible population mean")
ax.set_title(f"Path C (NeT-only) cluster profiles, k={SELECTED_K}, N={len(ne)}")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(FIG / "fig3_pathc_cluster_profiles.png", dpi=150)
plt.show()
print("Saved fig3_pathc_cluster_profiles.png")

## 3. Comparison to Phase 1 and Path A (Part 3 of blueprint -- the falsifiable prediction)

**Predicted in advance:** LOW ARI against both -- the mirror image of DS5's Path C, where NeT-only
clustering reproduced Phase 1 almost perfectly (ARI=0.990).

In [ ]:
overlap = ne.merge(phase1[["participant_id", "cluster_label"]], on="participant_id", suffixes=("_pathc", "_phase1"))
overlap = overlap.merge(patha[["participant_id", "cluster_label"]].rename(columns={"cluster_label": "cluster_label_patha"}), on="participant_id")
print(f"Overlap N: {len(overlap)}")

ari_phase1 = adjusted_rand_score(overlap["cluster_label_phase1"], overlap["cluster_label_pathc"])
ari_patha = adjusted_rand_score(overlap["cluster_label_patha"], overlap["cluster_label_pathc"])
print(f"ARI(Path C vs Phase 1): {ari_phase1:.3f}  (DS5's equivalent: 0.990)")
print(f"ARI(Path C vs Path A):  {ari_patha:.3f}")

if ari_phase1 < 0.15:
    verdict = "CONFIRMED: NeT-only clustering is close to orthogonal to Phase 1 -- consistent with Path A's finding that behavioral features, not network features, drive the Structured phenotype split. Mirror image of DS5."
elif ari_phase1 < 0.4:
    verdict = "PARTIAL: some correspondence, weaker than DS5's 0.990 but not fully orthogonal -- network structure contributes some but not most of the Phase 1 split."
else:
    verdict = "NOT CONFIRMED: NeT-only clustering substantially reproduces Phase 1, closer to DS5's pattern than expected -- reported as a genuine surprise given Path A's ARI=1.000 result, not reconciled away."
print()
print(verdict)

print()
print("Cross-tab (Phase 1 rows x Path C columns):")
print(pd.crosstab(overlap["cluster_label_phase1"], overlap["cluster_label_pathc"]))

ari_result = pd.DataFrame([
    {"comparison": "Path C vs Phase 1", "overlap_n": len(overlap), "ARI": round(ari_phase1, 4), "ds5_equivalent_ARI": 0.990, "verdict": verdict},
    {"comparison": "Path C vs Path A", "overlap_n": len(overlap), "ARI": round(ari_patha, 4), "ds5_equivalent_ARI": np.nan, "verdict": ""},
])
ari_result.to_csv(OUT / "net_only_vs_phase1_and_patha_ari.csv", index=False)
print("\nSaved: net_only_vs_phase1_and_patha_ari.csv")

## 4. Sub-clustering within Phase 1 Profile A (N=22) and Profile B (N=14) (Part 2 of blueprint)

`min_cell` floors scaled proportionally from Phase 1's own rule (min_cell>=10 for N=36, ratio~0.28):
**min_cell>=6 for Profile A (N=22), min_cell>=4 for Profile B (N=14).** Stated in the blueprint before
running, not chosen after seeing results.

In [ ]:
def subcluster(sub_df, min_cell, label):
    Xs = sub_df[feature_cols].to_numpy()
    Xs_z = (Xs - Xs.mean(axis=0)) / Xs.std(axis=0)
    km = KMeans(n_clusters=2, n_init=10, random_state=RANDOM_STATE).fit(Xs_z)
    sizes = pd.Series(km.labels_).value_counts()
    sil = silhouette_score(Xs_z, km.labels_)
    print(f"{label} (N={len(sub_df)}): k=2, sizes={dict(sizes)}, silhouette={sil:.3f}")
    if sizes.min() < min_cell:
        print(f"  -> INFEASIBLE (min_cell={sizes.min()} < {min_cell})")
        return {"group": label, "n": len(sub_df), "feasible": False,
                "min_cell_floor": min_cell, "sizes": dict(sizes), "silhouette": round(sil,3),
                "jaccard_mean": np.nan, "jaccard_std": np.nan}
    jm, js = bootstrap_jaccard(Xs_z, km.labels_, 2, n_boot=100, frac=0.8, seed=RANDOM_STATE)
    verdict = "STABLE" if jm >= 0.75 else "MARGINAL/TENTATIVE" if jm >= 0.60 else "UNSTABLE"
    print(f"  -> feasible, bootstrap Jaccard={jm:.3f} ({verdict}) -- treat as tentative regardless, per blueprint Part 2")
    return {"group": label, "n": len(sub_df), "feasible": True,
            "min_cell_floor": min_cell, "sizes": dict(sizes), "silhouette": round(sil,3),
            "jaccard_mean": round(jm,3), "jaccard_std": round(js,3)}

profileA = ne[ne["participant_id"].isin(phase1[phase1.cluster_label == "Profile A"].participant_id)]
profileB = ne[ne["participant_id"].isin(phase1[phase1.cluster_label == "Profile B"].participant_id)]
print(f"Profile A NeT-eligible N={len(profileA)}, Profile B NeT-eligible N={len(profileB)}")
print()

result_A = subcluster(profileA, min_cell=6, label="Profile A NeT sub-cluster")
result_B = subcluster(profileB, min_cell=4, label="Profile B NeT sub-cluster")

pd.DataFrame([result_A]).to_csv(OUT / "profileA_net_subcluster.csv", index=False)
pd.DataFrame([result_B]).to_csv(OUT / "profileB_net_subcluster.csv", index=False)
print("\nSaved: profileA_net_subcluster.csv, profileB_net_subcluster.csv")

## 5. model_run_summary.md

In [ ]:
summary = f"""# Structured Path C -- Network-Substructure -- Model Run Summary

**Run date:** 2026-08-05
**Notebook:** notebooks/121_structured_network_subphenotype.ipynb
**Blueprint:** outputs/reports/structured/structured_path_c_blueprint.md

## Population and features
- Full-sample NeT-only: N={len(ne)} (is_net_eligible)
- Features: HHI_net, rho_self, H_trans, H_state_marginal (top1_share_net excluded, r=0.961 with HHI_net)

## K-selection
"""
for k in k_results:
    r = k_results[k]
    summary += f"- k={k}: silhouette={r['silhouette']:.3f}, CH={r['CH']:.1f}, DB={r['DB']:.3f}, min_cell={r['min_size']}, sizes={r['sizes']}\n"
summary += f"""
**Selected k={SELECTED_K}** (silhouette={k_results[SELECTED_K]['silhouette']:.3f})

## Stability
- Bootstrap Jaccard: mean={jac_mean:.3f}, std={jac_std:.3f} -> {stability_verdict}
- PC1+PC2 variance explained: {pca.explained_variance_ratio_[:2].sum()*100:.1f}%

## Cluster profiles (raw means)
{ne.groupby("cluster_label")[feature_cols].mean().round(4).to_string()}

## Comparison to Phase 1 and Path A (the central Path C question)
- Overlap N: {len(overlap)}
- ARI(Path C vs Phase 1): {ari_phase1:.3f}  (DS5's equivalent: 0.990)
- ARI(Path C vs Path A): {ari_patha:.3f}
- {verdict}

## Sub-clustering within Phase 1 profiles
- Profile A (N={len(profileA)}, min_cell floor=6): {result_A}
- Profile B (N={len(profileB)}, min_cell floor=4): {result_B}

## Key limitations
1. Sub-cluster N (14-22) is far below DS5's already-tentative Profile A sub-clustering (N=125)
2. H_trans/H_state_marginal correlation (r=0.715) retained per Phase 1 precedent, not resolved
3. H_pi (stationary distribution entropy) excluded entirely -- only 8/61 stationary-eligible, far too
   thin to use even as a supplementary check at this population size
"""

with open(OUT / "model_run_summary.md", "w") as f:
    f.write(summary)

print(summary)